# ElasticMind — Workload EDA

Exploratory analysis of the **simulated** workload traces used for training
the ElasticMind forecaster. All numbers below are simulated.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd
from ml.data.generate import default_events, generate_workload
from ml.data.loader import trace_summary

frames = {}
for profile in ["steady", "diurnal", "spiky"]:
    events = default_events(profile, 14)
    frames[profile] = generate_workload(profile=profile, days=14, seed=42, events=events)

pd.DataFrame({p: trace_summary(df) for p, df in frames.items()}).T

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=False)
for ax, (profile, df) in zip(axes, frames.items(), strict=False):
    ax.plot(df["timestamp"], df["requests"], lw=0.8)
    ax.set_title(f"{profile} — simulated requests/min (14 days)")
    ax.set_ylabel("req/min")
plt.tight_layout()
plt.show()

In [ ]:
df = frames["diurnal"]
hod = df.groupby(df["timestamp"].dt.hour)["requests"].mean()
dow = df.groupby(df["timestamp"].dt.dayofweek)["requests"].mean()

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
hod.plot(ax=ax[0], marker="o").set_title("Mean req/min by hour-of-day")
dow.plot(ax=ax[1], marker="o").set_title("Mean req/min by day-of-week (0=Mon)")
plt.tight_layout()
plt.show()

In [ ]:
df["requests"].describe(percentiles=[0.5, 0.9, 0.99]).to_frame().T

### Findings
- `diurnal`: strong 24 h cycle + weekend dip → cyclical hour/dow features matter.
- `spiky`: burst micro-spikes + injected flash spike / regime shift → good stress
  test for misprediction handling (Prompt 9).
- `steady`: flat baseline used for policy unit tests.

All series above are **simulated**.